In [ ]:
!pip install torch transformers datasets scikit-learn


In [ ]:
!pip install kaggle



In [ ]:
!mkdir -p ~/.kaggle

In [ ]:
!cp kaggle.json ~/.kaggle/

cp: cannot stat 'kaggle.json': No such file or directory


In [ ]:
from google.colab import files

uploaded = files.upload()  # This will open a file picker


Saving kaggle.json to kaggle.json


In [ ]:
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets list


ref                                                              title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
wardabilal/spotify-global-music-dataset-20092025                 Spotify Global Music Dataset (2009–2025)               1289021  2025-11-11 09:43:05.933000          16387        395  1.0              
neurocipher/heartdisease                                         Heart Disease                                             3491  2025-12-11 15:29:14.327000           2114         57  1.0              
kundanbedmutha/exam-score-prediction-dataset                     Exam Score Prediction Dataset                           325454  2025-11-28 07:29:01.047000           5863        127  1.0          

In [ ]:
!kaggle datasets download -d debarshichanda/goemotions
!unzip goemotions.zip

Dataset URL: https://www.kaggle.com/datasets/debarshichanda/goemotions
License(s): CC0-1.0
  0% 0.00/17.6M [00:00<?, ?B/s]
100% 17.6M/17.6M [00:00<00:00, 1.03GB/s]
Archive:  goemotions.zip
  inflating: GoEmotionsFormat.PNG    
  inflating: README.md               
  inflating: analyze_data.py         
  inflating: calculate_metrics.py    
  inflating: data/dev.tsv            
  inflating: data/ekman_labels.csv   
  inflating: data/ekman_mapping.json  
  inflating: data/emotions.txt       
  inflating: data/full_dataset/goemotions_1.csv  
  inflating: data/full_dataset/goemotions_2.csv  
  inflating: data/full_dataset/goemotions_3.csv  
  inflating: data/sentiment_dict.json  
  inflating: data/sentiment_mapping.json  
  inflating: data/test.tsv           
  inflating: data/train.tsv          
  inflating: extract_words.py        
  inflating: goemotions_model_card.pdf  
  inflating: plots/colors.tsv        
  inflating: plots/correlations.pdf  
  inflating: plots/hierarchical_clustering

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn


In [ ]:
import pandas as pd
import numpy as np

from datasets import Dataset
from sklearn.preprocessing import MultiLabelBinarizer

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import f1_score, accuracy_score


In [ ]:
df_train = pd.read_csv("data/train.tsv", sep="\t", header=None, names=["text", "labels", "id"])
df_val   = pd.read_csv("data/dev.tsv",   sep="\t", header=None, names=["text", "labels", "id"])
df_test  = pd.read_csv("data/test.tsv",  sep="\t", header=None, names=["text", "labels", "id"])

print(df_train.head())


                                                text labels       id
0  My favourite food is anything I didn't have to...     27  eebbqej
1  Now if he does off himself, everyone will thin...     27  ed00q6i
2                     WHY THE FUCK IS BAYLESS ISOING      2  eezlygj
3                        To make her feel threatened     14  ed7ypvh
4                             Dirty Southern Wankers      3  ed0bdzj


In [ ]:
for df in (df_train, df_val, df_test):
    df["text"] = df["text"].astype(str).str.strip()


In [ ]:
def parse_labels(label_str):
    s = str(label_str).strip()
    if s == "":
        return []
    return [int(x) for x in s.split(",")]

df_train["label_list"] = df_train["labels"].apply(parse_labels)
df_val["label_list"]   = df_val["labels"].apply(parse_labels)
df_test["label_list"]  = df_test["labels"].apply(parse_labels)


In [ ]:
num_labels = 28  # GoEmotions has 28 emotion labels (0–27)

mlb = MultiLabelBinarizer(classes=list(range(num_labels)))
y_train = mlb.fit_transform(df_train["label_list"])
y_val   = mlb.transform(df_val["label_list"])
y_test  = mlb.transform(df_test["label_list"])

y_train.shape, y_val.shape, y_test.shape


((43410, 28), (5426, 28), (5427, 28))

In [ ]:
train_dataset = Dataset.from_pandas(
    df_train[["text"]].assign(labels=y_train.astype(np.float32).tolist())
)
val_dataset = Dataset.from_pandas(
    df_val[["text"]].assign(labels=y_val.astype(np.float32).tolist())
)
test_dataset = Dataset.from_pandas(
    df_test[["text"]].assign(labels=y_test.astype(np.float32).tolist())
)

train_dataset[0]

{'text': "My favourite food is anything I didn't have to cook myself.",
 'labels': [0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  1.0]}

In [ ]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_enc = train_dataset.map(tokenize_batch, batched=True)
val_enc   = val_dataset.map(tokenize_batch, batched=True)
test_enc  = test_dataset.map(tokenize_batch, batched=True)


Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [ ]:
cols = ["input_ids", "attention_mask", "labels"]

train_enc.set_format(type="torch", columns=cols)
val_enc.set_format(type="torch", columns=cols)
test_enc.set_format(type="torch", columns=cols)


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # sigmoid for multi-label
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    # exact match: all labels correct
    exact_match = accuracy_score(labels, preds) # Changed this line

    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "exact_match_acc": exact_match
    }

In [ ]:
!pip install -U transformers

import transformers
print(transformers.__version__)


4.57.3


In [ ]:
from transformers import TrainingArguments, Trainer   # <--- REQUIRED IMPORT

training_args = TrainingArguments(
    output_dir="distilbert-goemotions",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    logging_steps=100,
)

In [ ]:
trainer = Trainer(
    model=model,              # DistilBERT/BERT model
    args=training_args,
    train_dataset=train_enc,  # tokenized train dataset
    eval_dataset=val_enc,     # tokenized validation dataset
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,  # the f1_micro / f1_macro function we defined
)


/tmp/ipython-input-3153113259.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
import torch
print(torch.cuda.is_available())
torch.cuda.get_device_name(0)


True


'Tesla T4'

In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Exact Match Acc
1,0.086400,0.086604,0.558273,0.348414,0.440472
2,0.078900,0.083568,0.566966,0.409608,0.448765


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Exact Match Acc
1,0.086400,0.086604,0.558273,0.348414,0.440472
2,0.078900,0.083568,0.566966,0.409608,0.448765
3,0.071600,0.083972,0.578072,0.420014,0.463325


TrainOutput(global_step=8142, training_loss=0.0837360029576657, metrics={'train_runtime': 1487.6714, 'train_samples_per_second': 87.539, 'train_steps_per_second': 5.473, 'total_flos': 4358276517476352.0, 'train_loss': 0.0837360029576657, 'epoch': 3.0})

In [ ]:
metrics_test = trainer.evaluate(test_enc)
metrics_test


{'eval_loss': 0.08324359357357025,
 'eval_f1_micro': 0.5846553483625568,
 'eval_f1_macro': 0.4250879465112994,
 'eval_exact_match_acc': 0.46821448313985625,
 'eval_runtime': 18.8446,
 'eval_samples_per_second': 287.987,
 'eval_steps_per_second': 9.021,
 'epoch': 3.0}

In [ ]:
predictions = trainer.predict(test_enc)
logits = predictions.predictions
labels = predictions.label_ids


In [ ]:
import numpy as np

probs = 1 / (1 + np.exp(-logits))   # sigmoid
preds = (probs > 0.5).astype(int)   # threshold


In [ ]:
exact_match = (preds == labels).all(axis=1).mean()
print("Exact Match Accuracy:", exact_match)


Exact Match Accuracy: 0.46821448313985625


In [ ]:
per_label_accuracy = (preds == labels).sum(axis=0) / len(labels)
print("Per-label Accuracy:", per_label_accuracy)

for i, acc in enumerate(per_label_accuracy):
    print(f"Label {i}: {acc:.3f}")


Per-label Accuracy: [0.94343099 0.98286346 0.96812235 0.94379952 0.94177262 0.97696702
 0.97401879 0.9515386  0.98765432 0.973466   0.94988023 0.98175788
 0.99318224 0.98452184 0.99170813 0.99004975 0.99889442 0.9800995
 0.98525889 0.99576193 0.97401879 0.99705178 0.97530864 0.9979731
 0.99207665 0.97917818 0.97899392 0.78570112]
Label 0: 0.943
Label 1: 0.983
Label 2: 0.968
Label 3: 0.944
Label 4: 0.942
Label 5: 0.977
Label 6: 0.974
Label 7: 0.952
Label 8: 0.988
Label 9: 0.973
Label 10: 0.950
Label 11: 0.982
Label 12: 0.993
Label 13: 0.985
Label 14: 0.992
Label 15: 0.990
Label 16: 0.999
Label 17: 0.980
Label 18: 0.985
Label 19: 0.996
Label 20: 0.974
Label 21: 0.997
Label 22: 0.975
Label 23: 0.998
Label 24: 0.992
Label 25: 0.979
Label 26: 0.979
Label 27: 0.786


In [ ]:
sample_accuracy = (preds == labels).sum(axis=1) / labels.shape[1]
print("Average per-sample label accuracy:", sample_accuracy.mean())


Average per-sample label accuracy: 0.9705375240201112


In [ ]:
from sklearn.metrics import hamming_loss
print("Hamming Loss:", hamming_loss(labels, preds))


Hamming Loss: 0.029462475979888915


In [ ]:
# Load emotion names in index order (0..27)
with open("data/emotions.txt", "r") as f:
    EMOTION_NAMES = [line.strip() for line in f.readlines()]

len(EMOTION_NAMES), EMOTION_NAMES[:10]


(28,
 ['admiration',
  'amusement',
  'anger',
  'annoyance',
  'approval',
  'caring',
  'confusion',
  'curiosity',
  'desire',
  'disappointment'])

In [ ]:
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_emotions(text, threshold=0.5, top_k=None):
    model.eval()

    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=64
    ).to(device)

    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits.squeeze(0).cpu().numpy()

    # Sigmoid for multi-label
    probs = 1 / (1 + np.exp(-logits))

    # Two modes:
    # 1) threshold-based (all emotions > threshold)
    # 2) top_k (highest k emotions)

    if top_k is not None:
        # Get top_k emotions regardless of threshold
        idx_sorted = np.argsort(probs)[::-1][:top_k]
        result = [(EMOTION_NAMES[i], float(probs[i])) for i in idx_sorted]
    else:
        # Threshold-based: show all emotions above threshold
        idx_active = [i for i, p in enumerate(probs) if p >= threshold]
        result = [(EMOTION_NAMES[i], float(probs[i])) for i in idx_active]
        # Sort by probability desc
        result.sort(key=lambda x: x[1], reverse=True)

    return result, probs


In [ ]:
sentences = [
    "I am really anxious but also a bit hopeful about the future."
]

for s in sentences:
    emotions, _ = predict_emotions(s, threshold=0.3)  # lower threshold to see more labels
    print(f"\nText: {s}")
    if not emotions:
        print("  → No strong emotions detected.")
    else:
        for name, score in emotions:
            print(f"  {name}: {score:.3f}")



Text: I am really anxious but also a bit hopeful about the future.
  optimism: 0.558


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
save_path = "/content/drive/MyDrive/bert_emotion_model"


In [ ]:
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)


NameError: name 'trainer' is not defined

In [ ]:
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


NameError: name 'model' is not defined

In [ ]:
import os
for f in os.listdir("/content/drive/MyDrive/bert_emotion_model"):
    print(f)

model.safetensors
training_args.bin
config.json
tokenizer.json
special_tokens_map.json
tokenizer_config.json
vocab.txt


### **FROM HERE**

In [ ]:
!pip install openai-whisper -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 2.8 MB/s eta 0:00:00


In [ ]:
!pip install fastapi uvicorn pyngrok groq transformers torch nest-asyncio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 6.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_path = "/content/drive/MyDrive/bert_emotion_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print(f"Model loaded on {device} ✅")

Mounted at /content/drive


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded on cpu ✅


In [ ]:
EMOTION_NAMES = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]

print(f"Total emotions: {len(EMOTION_NAMES)} ✅")

Total emotions: 28 ✅


In [ ]:
RESPONSE_BANK = {
    "admiration": [
        "It's wonderful that you're feeling inspired by someone or something.",
        "That sense of admiration says a lot about your appreciation for greatness.",
        "It's beautiful to recognize and celebrate what moves you."
    ],
    "amusement": [
        "It's great that something brought a smile to your face today!",
        "Laughter and joy are such powerful healers.",
        "Hold onto that lightness — it's a gift."
    ],
    "anger": [
        "It's completely valid to feel angry. Your feelings matter.",
        "Anger often signals that something important to you has been crossed.",
        "It takes courage to acknowledge anger rather than push it away."
    ],
    "annoyance": [
        "It's okay to feel frustrated — small things can wear us down too.",
        "Your irritation is valid. Sometimes things just get under our skin.",
        "Take a breath — your feelings are completely understandable."
    ],
    "approval": [
        "It sounds like something really resonated with you today.",
        "That positive feeling of agreement can be really grounding.",
        "It's good to recognize when things align with your values."
    ],
    "caring": [
        "Your compassion for others is a true strength.",
        "The fact that you care so deeply speaks volumes about who you are.",
        "Caring for others is meaningful — just remember to care for yourself too."
    ],
    "confusion": [
        "It's okay not to have all the answers right now.",
        "Confusion is often the first step toward clarity.",
        "Give yourself grace — not everything needs to make sense immediately."
    ],
    "curiosity": [
        "That curiosity of yours is such a wonderful quality.",
        "Asking questions and exploring is how we grow.",
        "Your openness to learning is something to be proud of."
    ],
    "desire": [
        "Wanting something deeply is a sign of what truly matters to you.",
        "It's okay to acknowledge what your heart is reaching for.",
        "Your desires are valid — they're part of who you are."
    ],
    "disappointment": [
        "I'm sorry things didn't go the way you hoped. That genuinely hurts.",
        "Disappointment is hard, especially when you cared so much.",
        "It's okay to grieve what you expected — your feelings are real."
    ],
    "disapproval": [
        "It's okay to feel unsettled when something doesn't sit right with you.",
        "Your instincts and values matter — trust them.",
        "Standing by what you believe in takes real courage."
    ],
    "disgust": [
        "It makes sense to feel repelled by things that conflict with your values.",
        "Your reaction shows how strongly you stand for what's right.",
        "It's okay to distance yourself from things that make you feel this way."
    ],
    "embarrassment": [
        "We've all been there — embarrassment fades, I promise.",
        "Being vulnerable enough to feel embarrassed means you care, and that's human.",
        "Be gentle with yourself. This moment doesn't define you."
    ],
    "excitement": [
        "That excitement is contagious — hold onto that energy!",
        "It's so good to feel alive and energized about something.",
        "Let that enthusiasm carry you forward!"
    ],
    "fear": [
        "It's okay to be scared. Fear means something matters to you.",
        "You don't have to face your fears alone.",
        "Acknowledging fear is the first brave step through it."
    ],
    "gratitude": [
        "That sense of gratitude is such a powerful thing to carry.",
        "Recognizing the good around you is a beautiful practice.",
        "Gratitude has a quiet way of healing us from the inside."
    ],
    "grief": [
        "I'm so sorry for your loss. Grief is love with nowhere to go.",
        "There's no right way to grieve — be gentle with yourself.",
        "Your pain is valid. Take all the time you need."
    ],
    "joy": [
        "That joy is so precious — soak every bit of it in.",
        "Happiness looks good on you. Celebrate this moment.",
        "It's wonderful to feel this kind of lightness."
    ],
    "love": [
        "Love is one of the most powerful things we can feel.",
        "That warmth in your heart is something truly special.",
        "Cherish that feeling — it's one of life's greatest gifts."
    ],
    "nervousness": [
        "Nerves show that you care about what's ahead — that's not a bad thing.",
        "It's okay to feel anxious. Take it one breath at a time.",
        "You've gotten through hard things before. You can do this too."
    ],
    "optimism": [
        "That hope you're carrying is powerful — don't let go of it.",
        "Optimism is a form of courage. Keep looking forward.",
        "Believing in better days is half the battle."
    ],
    "pride": [
        "You should absolutely feel proud — you've earned it.",
        "Recognizing your own achievements is so important.",
        "That pride is well deserved. Celebrate yourself!"
    ],
    "realization": [
        "Those moments of clarity can be life changing.",
        "It takes real self-awareness to have a breakthrough like that.",
        "Sit with that realization — it came to you for a reason."
    ],
    "relief": [
        "That weight lifting off your shoulders is such a good feeling.",
        "Relief is your mind and body saying — you made it through.",
        "Take a deep breath and let yourself feel that ease."
    ],
    "remorse": [
        "The fact that you feel remorse shows how much you care.",
        "Be kind to yourself — growth often comes through mistakes.",
        "Acknowledging regret takes courage. Forgive yourself too."
    ],
    "sadness": [
        "I'm really sorry you're feeling this way. You don't have to go through it alone.",
        "It's okay to feel sad — let yourself feel it without judgment.",
        "Your pain is valid, and it's okay to not be okay right now."
    ],
    "surprise": [
        "Life has a way of catching us off guard sometimes!",
        "Unexpected moments can shake us up — how are you feeling about it?",
        "It's okay to take a moment to process something surprising."
    ],
    "neutral": [
        "I'm here to listen, whatever's on your mind.",
        "Sometimes just talking helps — feel free to share anything.",
        "I'm with you. Take your time."
    ]
}

import random

def get_template(emotions):
    if not emotions:
        return random.choice(RESPONSE_BANK["neutral"])
    # Pick template based on top detected emotion
    top_emotion = emotions[0][0]
    templates = RESPONSE_BANK.get(top_emotion, RESPONSE_BANK["neutral"])
    return random.choice(templates)

print("Response bank loaded ✅")

Response bank loaded ✅


In [ ]:
import numpy as np

def predict_emotions(text, threshold=0.3):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=64
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits.squeeze(0).cpu().numpy()

    probs = 1 / (1 + np.exp(-logits))
    results = [(EMOTION_NAMES[i], float(probs[i])) for i in range(len(probs)) if probs[i] >= threshold]
    results.sort(key=lambda x: x[1], reverse=True)
    return results

In [ ]:
import whisper
whisper_model = whisper.load_model("base")
print("Whisper loaded ✅")

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 91.6MiB/s]


Whisper loaded ✅


In [ ]:
import tempfile
import os
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class HistoryMessage(BaseModel):
    role: str
    content: str

class MessageRequest(BaseModel):
    message: str
    history: Optional[List[HistoryMessage]] = []

CRISIS_EMOTIONS = {"grief", "fear", "sadness", "nervousness", "remorse"}
CRISIS_THRESHOLD = 0.6

@app.post("/transcribe")
async def transcribe(audio: UploadFile = File(...)):
    with tempfile.NamedTemporaryFile(delete=False, suffix=".webm") as tmp:
        tmp.write(await audio.read())
        tmp_path = tmp.name
    try:
        with open(tmp_path, "rb") as f:
            result = groq_client.audio.transcriptions.create(
                model="whisper-large-v3-turbo",
                file=f,
                response_format="verbose_json",  # gives us language info too
            )
        return {
            "transcript": result.text.strip(),
            "language": result.language  # bonus — detected language
        }
    finally:
        os.unlink(tmp_path)

@app.post("/chat")
async def chat(req: MessageRequest):
    emotions = predict_emotions(req.message)
    response = generate_response(req.message, emotions, req.history)

    crisis = any(
        name in CRISIS_EMOTIONS and score >= CRISIS_THRESHOLD
        for name, score in emotions
    )

    return {
        "response": response,
        "emotions": emotions,
        "crisis": crisis
    }

@app.get("/")
async def root():
    return {"status": "running ✅"}

In [ ]:
from groq import Groq
import random

GROQ_API_KEY = "YOUR_API_KEY"  # 👈 paste your key here
groq_client = Groq(api_key=GROQ_API_KEY)

def generate_response(user_message, emotions, history=[]):
    template = get_template(emotions)
    emotion_str = ", ".join([f"{e[0]} ({e[1]:.0%})" for e in emotions]) if emotions else "neutral"

    messages = [
        {
            "role": "system",
            "content": f"""You are a compassionate mental health support chatbot.
The user is currently feeling: {emotion_str}.
Use this as your response foundation: "{template}"
Expand it naturally based on what the user actually said.
IMPORTANT: Detect the language of the user's message and respond in the SAME language.
Keep it warm, concise (2-3 sentences), and human. No medical advice."""
        }
    ]

    # Add conversation history
    for h in history:
        messages.append({"role": h.role, "content": h.content})

    # Add current message
    messages.append({"role": "user", "content": user_message})

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages
    )
    return response.choices[0].message.content

print("Hybrid response system ready ✅")

Hybrid response system ready ✅


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
!pip install huggingface_hub -q

from huggingface_hub import HfApi
api = HfApi()

api.upload_folder(
    folder_path="/content/drive/MyDrive/bert_emotion_model",
    repo_id="Junu0208/bert-emotion-model",
    repo_type="model"
)

print("Model uploaded ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n_model/model.safetensors:   0%|          |  575kB /  268MB            

  ...n_model/training_args.bin:   1%|          |  48.0B / 5.84kB            

Model uploaded ✅


In [ ]:
import subprocess
import threading
import time

def run_server():
    import nest_asyncio
    import uvicorn
    nest_asyncio.apply()
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    import asyncio
    asyncio.run(server.serve())

# Start server in background
t = threading.Thread(target=run_server)
t.daemon = True
t.start()
time.sleep(2)

# Start Cloudflare tunnel
process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Extract URL
for line in process.stdout:
    line = line.decode()
    if "trycloudflare.com" in line:
        url = line.strip().split()[-1]
        print(f"Public URL: {url}")
        break

INFO:     Started server process [4769]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Public URL: trycloudflare.com...


In [ ]:
import subprocess
import time

process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

for line in process.stdout:
    line = line.decode()
    if "trycloudflare.com" in line:
        # Extract just the URL
        import re
        urls = re.findall(r'https://\S+\.trycloudflare\.com', line)
        if urls:
            print(f"Your permanent URL: {urls[0]}")
            break

Your permanent URL: https://periodically-confident-fashion-mls.trycloudflare.com
